# NB07 - Gradient Flow & Vanishing Gradients
## Background
Training deep networks requires gradients to flow backward through many layers. The vanishing gradient problem (Hochreiter, 1991; Bengio et al., 1994) occurs when gradients shrink exponentially as they backpropagate, preventing early layers from learning. The exploding gradient problem is its mirror image. Batch Normalization (Ioffe & Szegedy, 2015) and Layer Normalization (Ba et al., 2016) dramatically alleviate these issues by normalizing activations. Residual connections (He et al., 2015) provide gradient highways that bypass normalization entirely.

## What You'll Learn
- Gradient norm tracking per layer during training
- How activation functions affect gradient flow (sigmoid vs ReLU vs GELU)
- The effect of BatchNorm and LayerNorm on gradient magnitudes
- Gradient clipping: max-norm and value clipping

## Why This Matters
Gradient norm monitoring is a standard diagnostic in LLM training dashboards. Sudden spikes in gradient norm often precede training instability. LayerNorm positioning (pre-norm vs post-norm) has measurable effects on training stability for deep transformers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# ── Activation functions and their derivatives ────────────────────────────
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def sigmoid_grad(x): s = sigmoid(x); return s * (1 - s)

def relu(x): return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)

def tanh_fn(x): return np.tanh(x)
def tanh_grad(x): return 1 - np.tanh(x)**2

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715*x**3)))
def gelu_grad(x, eps=1e-5):
    return (gelu(x + eps) - gelu(x - eps)) / (2 * eps)

# ── Simulate gradient flow through L layers ────────────────────────────────
def simulate_gradient_flow(activation_fn, activation_grad, n_layers=20,
                            n_units=64, n_steps=5, seed=42) -> np.ndarray:
    """Track gradient norms layer by layer."""
    np.random.seed(seed)
    weights = [np.random.randn(n_units, n_units) * np.sqrt(2/n_units)
               for _ in range(n_layers)]
    x = np.random.randn(n_units)

    # Forward pass: record pre-activations
    pre_acts = []
    h = x.copy()
    for W in weights:
        z = W @ h
        pre_acts.append(z)
        h = activation_fn(z)

    # Backward pass: track gradient norms
    grad_norms = []
    delta = np.ones(n_units)  # upstream gradient
    for i in range(n_layers - 1, -1, -1):
        dact = activation_grad(pre_acts[i])
        delta = delta * dact
        grad_norms.insert(0, np.linalg.norm(delta))
        delta = weights[i].T @ delta

    return np.array(grad_norms)

activations = {
    'Sigmoid': (sigmoid, sigmoid_grad),
    'Tanh':    (tanh_fn, tanh_grad),
    'ReLU':    (relu, relu_grad),
    'GELU':    (gelu, gelu_grad),
}

fig, ax = plt.subplots(figsize=(12, 6))
for name, (fn, gfn) in activations.items():
    norms = simulate_gradient_flow(fn, gfn, n_layers=20)
    ax.semilogy(norms, '-o', markersize=3, label=name, linewidth=1.5)

ax.set_title('Gradient Norm per Layer (Layer 0 = input, Layer 19 = output)')
ax.set_xlabel('Layer index (from input)'); ax.set_ylabel('Gradient norm (log scale)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_07_gradient_flow.png', dpi=100, bbox_inches='tight')
plt.show()
print('Gradient flow comparison done.')

In [ ]:
# ── LayerNorm effect on gradient flow ─────────────────────────────────────
def layer_norm(x, eps=1e-5):
    mu = np.mean(x)
    sigma = np.std(x)
    return (x - mu) / (sigma + eps)

def simulate_with_layernorm(n_layers=20, n_units=64, seed=42):
    np.random.seed(seed)
    weights = [np.random.randn(n_units, n_units) * np.sqrt(2/n_units)
               for _ in range(n_layers)]
    x = np.random.randn(n_units)

    # Forward with LayerNorm before activation
    pre_acts, ln_inputs = [], []
    h = x.copy()
    for W in weights:
        z = W @ h
        z_ln = layer_norm(z)  # LayerNorm
        pre_acts.append(z_ln)
        ln_inputs.append(z)
        h = relu(z_ln)

    # Backward
    grad_norms = []
    delta = np.ones(n_units)
    for i in range(n_layers - 1, -1, -1):
        dact = relu_grad(pre_acts[i])
        # LayerNorm gradient (simplified: normalizes the delta too)
        delta_norm = delta * dact
        delta_norm = (delta_norm - np.mean(delta_norm)) / (np.std(delta_norm) + 1e-5)
        grad_norms.insert(0, np.linalg.norm(delta_norm))
        delta = weights[i].T @ delta_norm

    return np.array(grad_norms)

norms_relu   = simulate_gradient_flow(relu, relu_grad, n_layers=20)
norms_ln     = simulate_with_layernorm(n_layers=20)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(norms_relu, '-o', markersize=3, label='ReLU (no norm)', linewidth=1.5)
ax.semilogy(norms_ln, '-s', markersize=3, label='ReLU + LayerNorm', linewidth=1.5)
ax.set_title('Effect of LayerNorm on Gradient Flow')
ax.set_xlabel('Layer index'); ax.set_ylabel('Gradient norm (log scale)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_07_layernorm_effect.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Gradient clipping ─────────────────────────────────────────────────────
def clip_by_norm(grads: np.ndarray, max_norm: float = 1.0) -> np.ndarray:
    total_norm = np.linalg.norm(grads)
    if total_norm > max_norm:
        return grads * (max_norm / total_norm)
    return grads

def clip_by_value(grads: np.ndarray, clip_val: float = 1.0) -> np.ndarray:
    return np.clip(grads, -clip_val, clip_val)

# Simulate exploding gradient scenario
np.random.seed(0)
n_steps = 200
spikes = np.random.randn(n_steps)
spikes[40] = 50.0   # gradient spike at step 40
spikes[120] = -80.0 # another spike at step 120

clipped_norm  = [clip_by_norm(np.array([g]), 5.0)[0] for g in spikes]
clipped_value = [clip_by_value(np.array([g]), 5.0)[0] for g in spikes]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(spikes, alpha=0.7, label='Raw gradients', color='red')
ax.plot(clipped_norm, alpha=0.8, label='Clip by norm (max=5)', linewidth=1.5)
ax.plot(clipped_value, alpha=0.8, label='Clip by value (+-5)', linewidth=1.5, linestyle='--')
ax.axhline(5, color='gray', linestyle=':', alpha=0.5)
ax.axhline(-5, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Gradient Clipping: Raw vs Clipped')
ax.set_xlabel('Training step'); ax.set_ylabel('Gradient value')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_07_gradient_clipping.png', dpi=100, bbox_inches='tight')
plt.show()
print('Gradient clipping demo complete.')
print()
print('Standard practice for LLM training:')
print('  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)')
print('  Applied before optimizer.step(), after loss.backward()')